In [20]:
# ===========================================
# STEP 1 — Mount Google Drive
# ===========================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# ===========================================
# STEP 2 — Check if all CSV files exist
# ===========================================
import os

paths = [
    "/content/drive/My Drive/Cybersecurity_DDoS/ddos_val_data.csv",
    "/content/drive/My Drive/Cybersecurity_DDoS/ddos_val_embeddings.csv",
    "/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_val_data.csv",
    "/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_val_embeddings.csv",
    "/content/drive/My Drive/malware_project/val_374Features.csv",
    "/content/drive/My Drive/malware_project/val_embeddings_374Features.csv"
]

for p in paths:
    print(p, "exists?" , os.path.exists(p))


/content/drive/My Drive/Cybersecurity_DDoS/ddos_val_data.csv exists? True
/content/drive/My Drive/Cybersecurity_DDoS/ddos_val_embeddings.csv exists? True
/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_val_data.csv exists? True
/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_val_embeddings.csv exists? True
/content/drive/My Drive/malware_project/val_374Features.csv exists? True
/content/drive/My Drive/malware_project/val_embeddings_374Features.csv exists? True


In [22]:
import numpy as np
import pandas as pd

# ===========================================
# STEP 3 — Load Validation CSV Files
# ===========================================

# DDoS
ddos_val = pd.read_csv("/content/drive/My Drive/Cybersecurity_DDoS/ddos_val_data.csv")
ddos_val_emb = pd.read_csv("/content/drive/My Drive/Cybersecurity_DDoS/ddos_val_embeddings.csv")

# Zero-Day
zero_val = pd.read_csv("/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_val_data.csv")
zero_val_emb = pd.read_csv("/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_val_embeddings.csv")

# Malware
mal_val = pd.read_csv("/content/drive/My Drive/malware_project/val_374Features.csv")
mal_val_emb = pd.read_csv("/content/drive/My Drive/malware_project/val_embeddings_374Features.csv")

print("Validation Files Loaded Successfully!")


Validation Files Loaded Successfully!


In [23]:

# ===========================================
# STEP 4 — Prepare Validation Labels (Mapped)
# ===========================================


# =============================
# DDoS LABELS
# 0 -> BENIGN
# 1-4 stay 1-4
# =============================
ddos_labels = ddos_val["Label"].copy()

# =============================
# ZERO-DAY LABELS
# 0 -> BENIGN
# 1–14 → 5–18
# =============================
zero_labels = zero_val["Label"].copy()
zero_labels = zero_labels.replace({0: 0})
zero_labels = zero_labels.apply(lambda x: x + 4 if x > 0 else 0)

# =============================
# MALWARE LABELS (0–3 attacks, 4 benign)
# benign 4→0
# attacks 0–3 → 19–22
# =============================
mal_labels = mal_val["Class"].copy()
mal_labels = mal_labels.replace({4: 0})
mal_labels = mal_labels.apply(lambda x: x + 19 if x > 0 else 0)


In [24]:
# ===========================================
# STEP 5 — Convert Embeddings to NumPy Arrays
# ===========================================

ddos_emb_arr = ddos_val_emb.values
zero_emb_arr = zero_val_emb.values
mal_emb_arr  = mal_val_emb.values

print(ddos_emb_arr.shape, zero_emb_arr.shape, mal_emb_arr.shape)


zero_block = np.zeros((1, 128))

(71825, 128) (121303, 128) (2320, 128)


In [26]:
# ===========================================
# STEP 6 — Create 384-D Concatenated Embeddings
# ===========================================

# -------- DDoS Fused: [128 | 128zero | 128zero] --------
ddos_pad_zero1 = np.zeros((ddos_emb_arr.shape[0], 128))
ddos_pad_zero2 = np.zeros((ddos_emb_arr.shape[0], 128))

ddos_fused = np.concatenate([
    ddos_emb_arr,
    ddos_pad_zero1,
    ddos_pad_zero2
], axis=1)

# -------- Zero-Day Fused: [128zero | 128 | 128zero] --------

zero_pad_zero1 = np.zeros((zero_emb_arr.shape[0], 128))
zero_pad_zero2 = np.zeros((zero_emb_arr.shape[0], 128))

zero_fused = np.concatenate([
    zero_pad_zero1,
    zero_emb_arr,
    zero_pad_zero2
], axis=1)

# -------- Malware Fused: [128zero | 128zero | 128] --------

mal_pad_zero1 = np.zeros((mal_emb_arr.shape[0], 128))
mal_pad_zero2 = np.zeros((mal_emb_arr.shape[0], 128))

mal_fused = np.concatenate([
    mal_pad_zero1,
    mal_pad_zero2,
    mal_emb_arr
], axis=1)



In [27]:
# ===========================================
# STEP 7 — Join All Validation Embeddings + Labels
# ===========================================

X_val = np.vstack([ddos_fused, zero_fused, mal_fused])
y_val = np.concatenate([ddos_labels, zero_labels, mal_labels])

print("Final VAL Shape:", X_val.shape)
print("Final VAL Labels Shape:", y_val.shape)


Final VAL Shape: (195448, 384)
Final VAL Labels Shape: (195450,)


In [28]:
# Fix Label/Embedding Length Mismatch (Trim Extra Labels)

y_val = y_val[:len(X_val)]   # Trim extra 2 labels
print("After Fix:", X_val.shape, y_val.shape)


After Fix: (195448, 384) (195448,)


In [29]:

# ===========================================
# STEP 8 — Save Final Validation Embedding Dataset
# ===========================================

output_dir = "/content/drive/My Drive/Embedding_Concatenate"

# Convert to DataFrame
df_val = pd.DataFrame(X_val)
df_val["Label"] = y_val

# Save CSV
df_val.to_csv(f"{output_dir}/Concatenated_Val_embeddings.csv", index=False)

print(f"Saved successfully → {output_dir}/Concatenated_Val_embeddings.csv")


Saved successfully → /content/drive/My Drive/Embedding_Concatenate/Concatenated_Val_embeddings.csv
